# Stage-2 XAttn-Only StreamVLM with Llama 3.2 3B Instruct

This notebook runs the frozen-backbone stage-2 alignment setup:
- frozen EfficientNet 3D-CNN stream encoder
- frozen `meta-llama/Llama-3.2-3B-Instruct`
- train only inserted cross-attention layers

Before running it on a cluster or MacBook Pro, make sure the environment has a recent `transformers` version compatible with Llama 3.2 and that you have access to the gated Meta checkpoint.

Notes:
- MPS requires a recent Apple Silicon PyTorch build.
- `flash-attn` and `bitsandbytes` are not used by this stage-2 path and should not be assumed available on Mac.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists() and repo_root.parent.joinpath("src").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


In [ ]:
import sys
!{sys.executable} -m pip install huggingface_hub
from huggingface_hub import login
login(token="hf_XXXXXXXXXXXXXXXXXXXXXXXXX")

In [ ]:
import sys

!{sys.executable} -m pip install \
    accelerate==1.13.0 \
    huggingface_hub==1.8.0 \
    numpy==2.4.3 \
    opencv-python==4.13.0.92 \
    peft==0.18.1 \
    safetensors==0.7.0 \
    tokenizers==0.22.2 \
    torch==2.11.0 \
    torchaudio==2.11.0 \
    torchvision==0.26.0 \
    tqdm==4.67.3 \
    transformers==5.4.0

print("Done")



In [ ]:
from dataclasses import asdict

import torch
from torch.utils.data import DataLoader, Subset

from src.stage2 import (
    FrozenEfficientNetStreamEncoder,
    Stage2QEVDFit300KDataset,
    Stage2TrainingConfig,
    XAttnConfig,
    build_xattn_only_streamvlm,
    prepare_generation_inputs,
    prepare_training_batch,
    resolve_runtime,
    stage2_collate,
    train_stage2,
)

PROFILE = "cluster_a40" #"mac_m3_max"  # or "cluster_a40"
subset_size = 30000
val_subset_size = 128
subset_seed = 469
run_tag = "subset30000_e2_validated_instruct"
validation_frequency_steps = 2000
previous_invalid_run_dir = repo_root / "outputs" / "stage2_xattn_llama32_instruct_mac_m3_max_subset20000_e2"

profile_overrides = {
    "mac_m3_max": {
        "preferred_device": "mps",
        "num_workers": 0,
        "micro_batch_size": 1,
    },
    "cluster_a40": {
        "preferred_device": "cuda",
        "num_workers": 2,
        "micro_batch_size": 1,
    },
}

runtime = resolve_runtime(preferred_device=profile_overrides[PROFILE]["preferred_device"])

config = {
    "profile": PROFILE,
    "runtime": runtime,
    "data_root": repo_root / "data" / "combined",
    "split": "train",
    "llm_model_name_or_path": "meta-llama/Llama-3.2-3B-Instruct", 
    "vision_checkpoint_path": repo_root / "ckpts_efficientnet" / "fitness_ally_hypermodel" / "efficientnet4Lite_1.8.3.checkpoint",
    "output_dir": repo_root / "outputs" / f"stage2_xattn_llama32_{PROFILE}_{run_tag}",
    "num_workers": profile_overrides[PROFILE]["num_workers"],
}

xattn_config = XAttnConfig(
    adapter_insert_layers=(6, 8, 10, 12, 14, 16, 18, 20, 22),
    xattn_block_size=1,
    num_of_xattn_heads=1,
    attn_dim=None,
)

training_config = Stage2TrainingConfig(
    learning_rate=5e-6,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.95,
    grad_clip_norm=1.0,
    epochs=2,
    effective_batch_size=32,
    micro_batch_size=profile_overrides[PROFILE]["micro_batch_size"],
    log_every=10,
    eval_every_steps=validation_frequency_steps,
)

print("profile:", PROFILE)
print("runtime:", runtime)
print("run tag:", run_tag)
print("subset size:", subset_size)
print("validation subset size:", val_subset_size)
print("subset seed:", subset_seed)
print("validation frequency steps:", validation_frequency_steps)
print("previous invalid run dir:", previous_invalid_run_dir)
print(asdict(training_config))

## Native QEVD-FIT-300K layout

This notebook expects `data_root` to point at the native `combined/` folder:

```text
combined/
  short_clips/
  fine_grained_labels.json
  feedbacks_short_clips.json
  questions.json
```

In [ ]:
dataset = Stage2QEVDFit300KDataset(
    data_root=config["data_root"],
    split=config["split"],
)
original_dataset_size = len(dataset)
subset_generator = torch.Generator().manual_seed(subset_seed)
all_indices = torch.randperm(original_dataset_size, generator=subset_generator).tolist()
subset_size = min(subset_size, original_dataset_size)
remaining_after_train = max(0, original_dataset_size - subset_size)
val_subset_size = min(val_subset_size, remaining_after_train)
train_indices = all_indices[:subset_size]
val_indices = all_indices[subset_size : subset_size + val_subset_size]
train_subset = Subset(dataset, train_indices)
val_subset = Subset(dataset, val_indices)
trained_sample_idx = train_indices[0]
trained_sample = dataset.samples[trained_sample_idx]
probe_sample_idx = val_indices[0] if val_indices else train_indices[0]
probe_sample = dataset.samples[probe_sample_idx]
train_dataloader = DataLoader(
    train_subset,
    batch_size=training_config.micro_batch_size,
    shuffle=True,
    num_workers=config["num_workers"],
    collate_fn=stage2_collate,
)
validation_dataloader = DataLoader(
    val_subset,
    batch_size=training_config.micro_batch_size,
    shuffle=False,
    num_workers=config["num_workers"],
    collate_fn=stage2_collate,
)
train_val_overlap = len(set(train_indices) & set(val_indices))

print("original dataset size:", original_dataset_size)
print("training subset size:", len(train_subset))
print("validation subset size:", len(val_subset))
print("training dataloader batches:", len(train_dataloader))
print("validation dataloader batches:", len(validation_dataloader))
print("train/val overlap:", train_val_overlap)
print("dataset stats (full dataset):", dataset.stats)
print("trained sample index:", trained_sample_idx)
print("trained sample id:", trained_sample.sample_id)
print("probe sample index:", probe_sample_idx)
print("probe sample id:", probe_sample.sample_id)
print("probe sample path:", probe_sample.video_path)
print(train_subset[0])

In [ ]:
model = build_xattn_only_streamvlm(
    llm_model_name_or_path=config["llm_model_name_or_path"],
    device=config["runtime"].device,
    torch_dtype=config["runtime"].llm_dtype,
    allow_tokenizer_resize=False,  # first attempt; set True only if semantic token resolution fails
    xattn_config=xattn_config,
    hf_token="hf_XXXXXXXXXXXXXXXXXXXXXXXXX" #hf_XXXXXXXXXXXXXXXXXXXXXXXXX" # REDACTED 
)
vision_encoder = FrozenEfficientNetStreamEncoder(
    checkpoint_path=config["vision_checkpoint_path"],
    device=config["runtime"].device,
    torch_dtype=config["runtime"].vision_dtype,
)
initial_xattn_snapshot = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_trainable_parameters()
}

print("model class:", type(model.model))
print("config class:", type(model.model.config))
print("native xattn path:", getattr(model, "_uses_native_xattn", False))
print("model dtype:", next(model.model.parameters()).dtype)
print(model.special_token_strings)
print(model.special_token_ids)

In [ ]:
def audit_generation(model_to_eval, sample_to_eval, label):
    generation_inputs = prepare_generation_inputs(
        sample_to_eval,
        model=model_to_eval,
        vision_encoder=vision_encoder,
    )
    generated_ids = model_to_eval.generate_greedy(
        input_ids=generation_inputs["input_ids"],
        attention_mask=generation_inputs["attention_mask"],
        vision_feats=generation_inputs["vision_feats"],
        vision_xattn_mask=generation_inputs["vision_xattn_mask"],
        max_new_tokens=training_config.max_new_tokens_eval,
    )
    decoded = model_to_eval.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    print(f"{label} generation:")
    print(decoded)
    return decoded

model.zero_grad(set_to_none=True)
first_batch = next(iter(train_dataloader))
prepared = prepare_training_batch(first_batch, model=model, vision_encoder=vision_encoder)

print("runtime device:", config["runtime"].device)
print("runtime llm dtype:", config["runtime"].llm_dtype)
print("runtime vision dtype:", config["runtime"].vision_dtype)
for key in ["input_ids", "attention_mask", "vision_xattn_mask", "labels"]:
    print(key, prepared[key].shape)
print("vision feats:", prepared["vision_feats"]["feats"].shape)
label_tokens = prepared["labels"][0][prepared["labels"][0] != -100].tolist()
print("decoded supervised target:", model.tokenizer.decode(label_tokens, skip_special_tokens=False))

outputs = model(
    input_ids=prepared["input_ids"],
    attention_mask=prepared["attention_mask"],
    vision_feats=prepared["vision_feats"],
    vision_xattn_mask=prepared["vision_xattn_mask"],
    labels=prepared["labels"],
)
pretrain_loss = outputs.loss.detach()
pretrain_loss_isfinite = bool(torch.isfinite(pretrain_loss).item())
pretrain_logits_isfinite = bool(torch.isfinite(outputs.logits).all().item())
print("pre-train loss:", float(pretrain_loss))
print("pre-train finite loss:", pretrain_loss_isfinite)
print("pre-train finite logits:", pretrain_logits_isfinite)

outputs.loss.backward()
pretrain_grad_summary = []
for name, parameter in model.named_trainable_parameters():
    grad = parameter.grad
    if grad is None:
        continue
    grad_isfinite = bool(torch.isfinite(grad).all().item())
    max_abs = float(grad.abs().max().item()) if grad.numel() else 0.0
    pretrain_grad_summary.append((name, grad_isfinite, max_abs))
nonfinite_grad_names = [name for name, grad_isfinite, _ in pretrain_grad_summary if not grad_isfinite]
print("trainable parameter count:", len(initial_xattn_snapshot))
print("grad tensors checked:", len(pretrain_grad_summary))
print("non-finite grad tensors:", len(nonfinite_grad_names))
print(nonfinite_grad_names[:10])

model.zero_grad(set_to_none=True)
pretrain_generation_text = audit_generation(model, probe_sample, "pre-train probe")

In [ ]:
history = train_stage2(
    model=model,
    dataloader=train_dataloader,
    vision_encoder=vision_encoder,
    config=training_config,
    output_dir=config["output_dir"],
    validation_dataloader=validation_dataloader,
    probe_sample=probe_sample,
)
history

In [ ]:
posttrain_batch = next(iter(train_dataloader))
posttrain_prepared = prepare_training_batch(posttrain_batch, model=model, vision_encoder=vision_encoder)

with torch.no_grad():
    posttrain_outputs = model(
        input_ids=posttrain_prepared["input_ids"],
        attention_mask=posttrain_prepared["attention_mask"],
        vision_feats=posttrain_prepared["vision_feats"],
        vision_xattn_mask=posttrain_prepared["vision_xattn_mask"],
        labels=posttrain_prepared["labels"],
    )

posttrain_loss = posttrain_outputs.loss.detach()
posttrain_loss_isfinite = bool(torch.isfinite(posttrain_loss).item())
posttrain_logits_isfinite = bool(torch.isfinite(posttrain_outputs.logits).all().item())
print("post-train loss:", float(posttrain_loss))
print("post-train finite loss:", posttrain_loss_isfinite)
print("post-train finite logits:", posttrain_logits_isfinite)
posttrain_generation_text = audit_generation(model, probe_sample, "post-train probe")

In [ ]:
import json

model.save_stage2_checkpoint(
    config["output_dir"] / "final",
    extra_config={
        "training_config": asdict(training_config),
        "llm_model_name_or_path": config["llm_model_name_or_path"],
        "vision_checkpoint_path": str(config["vision_checkpoint_path"]),
    },
)
print("audited run dir:", config["output_dir"])
print("previous invalid run dir exists:", previous_invalid_run_dir.exists())

checkpoint_dirs = [
    config["output_dir"] / "epoch_1",
    config["output_dir"] / "epoch_2",
    config["output_dir"] / "final",
]
checkpoint_audit = []
final_checkpoint_state = None

for checkpoint_dir in checkpoint_dirs:
    record = {"checkpoint": checkpoint_dir.name, "exists": checkpoint_dir.exists()}
    if checkpoint_dir.exists():
        checkpoint_config = json.loads((checkpoint_dir / "stage2_config.json").read_text())
        state_dict = torch.load(checkpoint_dir / "xattn_state_dict.pt", map_location="cpu")
        nonfinite_tensors = [
            name for name, tensor in state_dict.items()
            if not torch.isfinite(tensor).all().item()
        ]
        max_abs_delta_vs_init = 0.0
        changed_tensor_count = 0
        for name, tensor in state_dict.items():
            init_tensor = initial_xattn_snapshot.get(name)
            if init_tensor is None:
                continue
            delta = float((tensor - init_tensor).abs().max().item())
            if delta > 0.0:
                changed_tensor_count += 1
            if delta > max_abs_delta_vs_init:
                max_abs_delta_vs_init = delta
        if checkpoint_dir.name == "final":
            final_checkpoint_state = state_dict

        record.update({
            "training_config": checkpoint_config.get("extra_config", {}).get("training_config"),
            "nonfinite_tensor_count": len(nonfinite_tensors),
            "nonfinite_tensors_preview": nonfinite_tensors[:10],
            "changed_tensor_count_vs_init": changed_tensor_count,
            "max_abs_delta_vs_init": max_abs_delta_vs_init,
        })
    checkpoint_audit.append(record)

checkpoint_audit

In [ ]:
import math

current_nonfinite_params = [
    name for name, parameter in model.named_trainable_parameters()
    if not torch.isfinite(parameter.detach()).all().item()
]
current_vs_final_max_delta = None
if final_checkpoint_state is not None:
    current_vs_final_max_delta = max(
        float((parameter.detach().cpu() - final_checkpoint_state[name]).abs().max().item())
        for name, parameter in model.named_trainable_parameters()
        if name in final_checkpoint_state
    )

def answer_segment(decoded_text):
    answer_begin = model.special_token_strings["answer_begin"]
    answer_end = model.special_token_strings["answer_end"]
    if answer_begin in decoded_text:
        decoded_text = decoded_text.split(answer_begin, 1)[1]
    if answer_end in decoded_text:
        decoded_text = decoded_text.split(answer_end, 1)[0]
    return decoded_text.strip()

def is_degenerate_answer(text):
    answer = answer_segment(text)
    if not answer:
        return True
    return answer.count("!") >= 16 or len(set(answer)) <= 3

history_all_nan = all(math.isnan(record["avg_loss"]) for record in history)
checkpoint_nonfinite = any(record.get("nonfinite_tensor_count", 0) > 0 for record in checkpoint_audit)

if not pretrain_loss_isfinite or not pretrain_logits_isfinite:
    audit_verdict = "Invalid run: model compatibility/integration"
elif checkpoint_nonfinite or history_all_nan or (not posttrain_loss_isfinite) or nonfinite_grad_names or current_nonfinite_params:
    audit_verdict = "Invalid run: numeric instability"
elif is_degenerate_answer(posttrain_generation_text):
    audit_verdict = "Valid but weak run"
else:
    audit_verdict = "Valid but weak run"

audit_summary = {
    "run_dir": str(config["output_dir"]),
    "subset_size": subset_size,
    "val_subset_size": val_subset_size,
    "epochs": training_config.epochs,
    "device": config["runtime"].device,
    "llm_dtype": str(config["runtime"].llm_dtype),
    "vision_dtype": str(config["runtime"].vision_dtype),
    "history_all_nan": history_all_nan,
    "pretrain_loss_isfinite": pretrain_loss_isfinite,
    "pretrain_logits_isfinite": pretrain_logits_isfinite,
    "nonfinite_grad_tensors": len(nonfinite_grad_names),
    "posttrain_loss_isfinite": posttrain_loss_isfinite,
    "posttrain_logits_isfinite": posttrain_logits_isfinite,
    "current_nonfinite_params": len(current_nonfinite_params),
    "current_vs_final_max_delta": current_vs_final_max_delta,
    "posttrain_answer_preview": answer_segment(posttrain_generation_text)[:200],
    "verdict": audit_verdict,
}
audit_summary